In [2]:
# 1) Download the CSV from Zenodo
import requests

url = "https://zenodo.org/records/5898311/files/vgsales.csv"
output_file = "vgsales.csv"   # ローカル保存

resp = requests.get(url)
with open(output_file, "wb") as f:
    f.write(resp.content)

print("Downloaded to:", output_file)

# 2) Load it into a DataFrame
import pandas as pd

df = pd.read_csv(output_file)
print("Shape:", df.shape)
df.head()

Downloaded to: vgsales.csv
Shape: (16598, 11)


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

In [4]:
#CSV読み込み
df = pd.read_csv("/content/sample_data/vgsales.csv")

df = df[['Year','Genre','Publisher','NA_Sales','EU_Sales','JP_Sales']]
df = df.dropna()

df.head()

,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales
0,2006.0,Sports,Nintendo,41.49,29.02,3.77
1,1985.0,Platform,Nintendo,29.08,3.58,6.81
2,2008.0,Racing,Nintendo,15.85,12.88,3.79
3,2009.0,Sports,Nintendo,15.75,11.01,3.28
4,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22


In [5]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

In [6]:
#データ読み込み
df = pd.read_csv("/content/sample_data/vgsales.csv")

df = df[['Year','Genre','Publisher','NA_Sales','EU_Sales','JP_Sales']]
df = df.dropna()

In [7]:
#年 × Genre × Publisher で売上集計
df = df.groupby(
    ['Year','Genre','Publisher']
)[['NA_Sales','EU_Sales','JP_Sales']].sum().reset_index()

In [8]:
#カテゴリを数値化
le_genre = LabelEncoder()
le_pub = LabelEncoder()

df['Genre'] = le_genre.fit_transform(df['Genre'])
df['Publisher'] = le_pub.fit_transform(df['Publisher'])

In [9]:
#時系列特徴量を作る（重要）過去売上を使う

df = df.sort_values(['Genre','Publisher','Year'])

df['NA_lag1'] = df.groupby(['Genre','Publisher'])['NA_Sales'].shift(1)
df['EU_lag1'] = df.groupby(['Genre','Publisher'])['EU_Sales'].shift(1)
df['JP_lag1'] = df.groupby(['Genre','Publisher'])['JP_Sales'].shift(1)

df = df.dropna()

In [10]:
#学習データ
features = [
    'Year',
    'Genre',
    'Publisher',
    'NA_lag1',
    'EU_lag1',
    'JP_lag1'
]

X = df[features]

y_na = df['NA_Sales']
y_eu = df['EU_Sales']
y_jp = df['JP_Sales']

In [11]:
#モデル作成
model_na = RandomForestRegressor(n_estimators=300)
model_eu = RandomForestRegressor(n_estimators=300)
model_jp = RandomForestRegressor(n_estimators=300)

model_na.fit(X,y_na)
model_eu.fit(X,y_eu)
model_jp.fit(X,y_jp)

RandomForestRegressor(n_estimators=300)

In [14]:
#未来予測（2035まで）
future_years = list(range(2026,2036))

In [15]:
#初期データ（最後の年）
last_data = df.sort_values('Year').groupby(
    ['Genre','Publisher']
).tail(1)

In [16]:
#逐次予測（時系列）
future_results = []

current = last_data.copy()

for year in future_years:

    current['Year'] = year

    X_future = current[features]

    current['NA_Sales'] = model_na.predict(X_future)
    current['EU_Sales'] = model_eu.predict(X_future)
    current['JP_Sales'] = model_jp.predict(X_future)

    future_results.append(current.copy())

    # 次年のlag更新
    current['NA_lag1'] = current['NA_Sales']
    current['EU_lag1'] = current['EU_Sales']
    current['JP_lag1'] = current['JP_Sales']

In [17]:
# 予測結果
future_df = pd.concat(future_results)

future_df['Genre'] = le_genre.inverse_transform(future_df['Genre'])
future_df['Publisher'] = le_pub.inverse_transform(future_df['Publisher'])

future_df.head()

# CSV保存
f = r'C:\py_analy_test\test_result.csv'
future_df.to_csv(f, index=False)